#Understanding the Data

##Patient and Demographic Info
1. ***`dep_name`*** – Name of the ED/department (useful if the data spans multiple hospital sites)
2. ***`esi`*** – Emergency Severity Index: a 1–5 triage urgency score assigned
at intake. 1 = most critical (resuscitation needed), 5 = least urgent (minor complaint). This is the standard triage acuity scale used in most U.S./North American EDs.
3. ***`age`***, ***`gender`*** – Age, gender
4. ***`ethnicity`***, ***`race`*** – Self-reported demographic categories (often separate columns because "ethnicity" and "race" are tracked as distinct fields in US health records, e.g., Hispanic/Non-Hispanic as ethnicity, separate from race categories)
5. ***`lang`*** – Patient's preferred/primary language
6. ***`religion`*** – Self-reported religious affiliation
7. ***`maritalstatus`*** – Married/single/divorced/etc.
8. ***`employstatus`*** – Employment status
9. ***`insurance_status`*** – Type of coverage (private, Medicaid, uninsured, etc.) — often a proxy for socioeconomic status in health equity research

##Visit and Arrival Info
1. ***`disposition`*** – What happened to the patient at the end of the ED visit: discharged home, admitted to hospital, transferred, left without being seen, etc. This is often the outcome variable for triage/prediction models.
2. ***`arrivalmode`*** – How the patient got to the ED: walk-in, ambulance, police, private vehicle, etc.
3. ***`arrivalmonth`***, ***`arrivalday`***, ***`arrivalhour_bin`*** – When they arrived (hour is "binned," meaning grouped into ranges like 0–4, 4–8, etc., rather than exact minute)
4. ***`previousdispo`*** – Disposition from a prior/previous ED visit (useful for tracking returning patients — a "bounce-back" indicator)

##Triage Vitals
1. ***`triage_vital_hr`*** – Heart rate (beats per minute)
2. ***`triage_vital_sbp`*** / dbp – Systolic / diastolic blood pressure (the "top" and "bottom" numbers)
3. ***`triage_vital_rr`*** – Respiratory rate (breaths per minute)
4. `***triage_vital_o2***` – Oxygen saturation (%), i.e., how well oxygenated the blood is
5. ***`triage_vital_o2_device`*** – Whether the O2 reading was on room air or with support (e.g., nasal cannula, mask)
6. ***`triage_vital_temp`*** `Inline code` – Body temperature
7. ***`triage_glucose`*** – Blood sugar level at triage (not always measured — only relevant for certain complaints like diabetes-related visits)

##Chief Complaint Flags
These flags are either yes(1) or no (0). They represent the chief complaint for a specific patient reported at triage. A single patient can have multiple flags.

#Top 10 Features That Indicate Triage Level

The following features are the indicators that are most predictive of triage level in no particular order:


1. Oxygen Saturation (***`triage_vital_o2`***)
2. Chief Complaint (each cc block)
3. Systolic Blood Pressure (***`triage_vital_sbp`***)
4. Heart Rate (***`triage_vital_hr`***)
5. Respiratory Rate (***`triage_vital_rr`***)
6. Temperature (***`triage_vital_temp`***)
7. Mode of Arrival (***`arrivalmode`***)
8. Age (***`age`***)
9. History (***`previousdispo`***)
10. Pain Complaints (cc blocks related to pain)

#Feasibility Study

Link to Feasibility Study: https://drive.google.com/file/d/1AZEI4BGekzZkg2yjZ5Rj-3O3Ijm6G_zq/view?usp=sharing

In [22]:
#Mounting Google Drive files
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted successfully!


In [23]:
#Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print ("Libraries imported successfully!")

Libraries imported successfully!


In [24]:
#Loading the CSV
FILEPATH = "/content/drive/MyDrive/yaleemmlc_admissionprediction_triage.csv"
df = pd.read_csv(FILEPATH)
print(f"Dataset Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

Dataset Loaded: 55121 rows x 226 columns
Columns: ['Unnamed: 0', 'dep_name', 'esi', 'age', 'gender', 'ethnicity', 'race', 'lang', 'religion', 'maritalstatus', 'employstatus', 'insurance_status', 'disposition', 'arrivalmode', 'arrivalmonth', 'arrivalday', 'arrivalhour_bin', 'previousdispo', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp', 'triage_vital_rr', 'triage_vital_o2', 'triage_vital_o2_device', 'triage_vital_temp', 'triage_glucose', 'cc_abdominalcramping', 'cc_abdominaldistention', 'cc_abdominalpain', 'cc_abdominalpainpregnant', 'cc_abnormallab', 'cc_abscess', 'cc_addictionproblem', 'cc_agitation', 'cc_alcoholintoxication', 'cc_alcoholproblem', 'cc_allergicreaction', 'cc_alteredmentalstatus', 'cc_animalbite', 'cc_ankleinjury', 'cc_anklepain', 'cc_anxiety', 'cc_arminjury', 'cc_armpain', 'cc_armswelling', 'cc_assaultvictim', 'cc_asthma', 'cc_backpain', 'cc_bleeding/bruising', 'cc_blurredvision', 'cc_bodyfluidexposure', 'cc_breastpain', 'cc_breathingdifficulty', 'cc_breath

In [25]:
#Looking at the first few rows
print(df.head(10))

   Unnamed: 0 dep_name  esi   age  gender           ethnicity  \
0           7        A  4.0  87.0  Female  Hispanic or Latino   
1          17        B  2.0  53.0    Male  Hispanic or Latino   
2          40        A  2.0  49.0  Female        Non-Hispanic   
3          47        A  3.0  22.0  Female  Hispanic or Latino   
4          60        A  2.0  62.0    Male        Non-Hispanic   
5          61        A  2.0  62.0    Male        Non-Hispanic   
6          86        B  4.0  38.0  Female  Hispanic or Latino   
7          87        A  3.0  39.0  Female  Hispanic or Latino   
8          88        B  3.0  39.0  Female  Hispanic or Latino   
9          95        C  3.0  26.0  Female        Non-Hispanic   

                 race     lang     religion      maritalstatus  ...  \
0               Other    Other  Pentecostal            Widowed  ...   
1               Other  English     Catholic  Significant Other  ...   
2  White or Caucasian  English     Catholic            Married  ...   


In [26]:
#Looking at the info of the data frame
print (df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55121 entries, 0 to 55120
Columns: 226 entries, Unnamed: 0 to cc_wristpain
dtypes: float64(210), int64(1), object(15)
memory usage: 95.0+ MB
None


In [27]:
#Looking if there are any missing values
columns = df.columns.tolist()
print(f"NaN count:\n{df[columns].isnull().sum()}")

NaN count:
Unnamed: 0               0
dep_name                 0
esi                      0
age                      0
gender                   0
                        ..
cc_woundcheck            0
cc_woundinfection        0
cc_woundre-evaluation    0
cc_wristinjury           0
cc_wristpain             0
Length: 226, dtype: int64


In [49]:
cc_flags = [c for c in df.columns if c.startswith("cc_")]

In [50]:
cc_flags = [c for c in df.columns if c.startswith("cc_")]

# Ordered (category -> keywords) rules. FIRST MATCH WINS, so specific systems come
# before the Trauma/MSK "pain/injury/swelling" near-catch-all, and Other is last.
CC_CATEGORIES = [
    ("Cardiovascular",             ["chestpain","chesttightness","palpitation","irregularheart","rapidheart","tachycard","hypertens","hypotens","cardiacarrest"]),
    ("Genitourinary/Renal",        ["dysuria","hematuria","urinary","flank","testicle","maleguproblem","femaleguproblem","groin","pelvic","vaginal","breastpain","std"]),
    ("Respiratory",                ["breathing","shortnessofbreath","dyspnea","cough","wheez","respiratory","asthma","hemoptysis","coldlike","nasalcongest","influenza","sinus","sorethroat","uri"]),
    ("Neurological",               ["headache","migraine","dizz","syncope","seizure","stroke","alteredmental","confusion","numbness","lossofconsciousness","unresponsive","lethargy","neurologic","hallucinat","extremityweakness"]),
    ("Gastrointestinal",           ["abdominal","epigastric","nausea","emesis","vomit","diarrhea","constipation","gibleeding","giproblem","rectal","swallowedforeign","ingestion","dehydration"]),
    ("Psych/Behavioral/Substance", ["anxiety","depress","agitation","suicid","homicid","psychot","psychiatric","panic","alcohol","drugproblem","drug/alcohol","addiction","detox","overdose","withdrawal","poison"]),
    ("ENT/Eye/Dental",             ["earpain","earproblem","otalgia","eye","conjunctiv","foreignbodyineye","blurredvision","dental","jaw","epistaxis","facial","oralswelling"]),
    ("Skin/Soft-tissue",           ["rash","cellulitis","abscess","cyst","skin","wound","mass","allergicreaction","edema","bruising"]),
    ("Endocrine/Metabolic/Heme",   ["bloodsugar","glycem","glucose","sicklecell"]),
    ("Constitutional/General",     ["fever","chills","fatigue","bodyache","generalizedbody","weakness"]),
    ("Trauma/Injury/MSK",          ["injury","fall","trauma","laceration","fracture","back","neck","joint","rib","assault","burn","animalbite","insectbite","pain","swelling"]),
]

def categorize(flag):
    stem = flag[3:]                     # drop the "cc_" prefix
    if "crash" in stem:                 # 'crash' contains 'rash' -> disambiguate FIRST
        return "Trauma/Injury/MSK"
    for category, keywords in CC_CATEGORIES:
        if any(k in stem for k in keywords):
            return category
    return "Other/Procedural/Admin"

# Build category -> list of flags, and confirm we placed every one.
from collections import defaultdict
groups = defaultdict(list)
for flag in cc_flags:
    groups[categorize(flag)].append(flag)

order = [name for name, _ in CC_CATEGORIES] + ["Other/Procedural/Admin"]
for name in order:
    print(f"{name:28s} {len(groups[name]):3d} flags")
print("-" * 40)
print(f"{'TOTAL placed':28s} {sum(len(v) for v in groups.values()):3d} of {len(cc_flags)}")




Cardiovascular                 9 flags
Genitourinary/Renal           17 flags
Respiratory                   15 flags
Neurological                  21 flags
Gastrointestinal              16 flags
Psych/Behavioral/Substance    18 flags
ENT/Eye/Dental                18 flags
Skin/Soft-tissue              14 flags
Endocrine/Metabolic/Heme       5 flags
Constitutional/General         8 flags
Trauma/Injury/MSK             49 flags
Other/Procedural/Admin        10 flags
----------------------------------------
TOTAL placed                 200 of 200
